# Notebook 2 — Introduction to Molecular Data: PDB Files
## CURE Laboratory: General Chemistry II | HTLV-1 Protease Inhibitor Discovery

**When to complete:** Week 4  
**Learning objectives:**
- Understand the PDB (Protein Data Bank) file format
- Parse and extract atomic coordinate data using Python
- Calculate distances between atoms and identify potential H-bond donors/acceptors
- Visualize a protein's atom-type composition
- Connect atomic coordinates to General Chemistry concepts (bond lengths, IMFs)

**Prerequisites:** Notebook 1 completed; familiarity with NumPy arrays and Matplotlib  
**Time estimate:** 90 minutes

---
> **What you'll work with:** The Protein Data Bank (PDB) is a freely available global archive of protein structures determined by X-ray crystallography, NMR, and cryo-electron microscopy. Every structure ever published is deposited here. The HTLV-1 protease structures you will simulate in Phase 2 came from the PDB. Today you will learn how to read and interpret PDB files — the universal language of structural biology.

---
## Part 1 — Setting Up and Fetching a PDB File


### 1.1 Install and import required libraries

We need one new library today: **`requests`**, which lets Python download files from the internet (like a web browser in code).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import requests
import io

mpl.rcParams['figure.dpi'] = 120
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False

print("Libraries loaded!")


### 1.2 Download a PDB file directly from the Protein Data Bank

We will use a small, well-characterized protein for this tutorial: **lysozyme** (PDB ID: 1LYZ). Lysozyme is an enzyme found in tears and saliva that breaks down bacterial cell walls. It is commonly used as a model system in MD simulation tutorials because it is small (129 residues), well-studied, and its behavior in simulation is well understood.

**Why lysozyme?** In Notebook 3, you will analyze an MD simulation of lysozyme. Studying the crystal structure first helps you understand what you are looking at in the simulation.

The PDB format stores atomic coordinates as plain text — each `ATOM` line gives the 3D (x, y, z) position of one atom. We will parse this file to extract useful information.


In [ ]:
# Download the lysozyme crystal structure from the Protein Data Bank
pdb_id = "1LYZ"
url = f"https://files.rcsb.org/download/{pdb_id}.pdb"

print(f"Downloading {pdb_id} from the Protein Data Bank...")
try:
    response = requests.get(url, timeout=10)
    pdb_text = response.text
    print(f"Download successful! File size: {len(pdb_text):,} characters")
    print("\nFirst 10 lines of the PDB file:")
    for line in pdb_text.split('\n')[:10]:
        print(line)
except Exception as e:
    print(f"Could not download — using embedded sample data instead.")
    print(f"(In a real lab session, the file will be provided on the course LMS.)")
    pdb_text = None


### 1.3 Understanding the PDB format

The PDB file stores information in fixed-width columns. Every `ATOM` record contains:

```
ATOM      1  N   ALA A   1      10.293  14.876   4.567  1.00 10.22           N
```

| Columns | Information |
|---|---|
| 1–6 | Record type (ATOM, HETATM, REMARK...) |
| 7–11 | Atom serial number |
| 13–16 | Atom name (N, CA, C, O, CB...) |
| 18–20 | Residue name (ALA, GLY, ASP...) |
| 22 | Chain ID (A, B...) |
| 23–26 | Residue sequence number |
| 31–38 | X coordinate (Å) |
| 39–46 | Y coordinate (Å) |
| 47–54 | Z coordinate (Å) |
| 55–60 | Occupancy |
| 61–66 | Temperature factor (B-factor) |
| 77–78 | Element symbol |

> **Chemistry connection:** The (x, y, z) coordinates are in **Angstroms** (1 Å = 10⁻¹⁰ m = 0.1 nm). Typical bond lengths: C–C ≈ 1.54 Å, C=O ≈ 1.23 Å, N–H ≈ 1.01 Å. The average hydrogen bond length (donor to acceptor) is 2.8–3.2 Å.


In [ ]:
def parse_pdb(pdb_text):
    """
    Parse a PDB file and return a DataFrame of atomic coordinates.
    Each row = one atom; columns = name, residue, chain, x, y, z, etc.
    """
    records = []
    for line in pdb_text.split('\n'):
        if line.startswith('ATOM') or line.startswith('HETATM'):
            try:
                record_type = line[0:6].strip()
                atom_serial = int(line[6:11])
                atom_name   = line[12:16].strip()
                residue_name= line[17:20].strip()
                chain_id    = line[21:22].strip()
                residue_num = int(line[22:26])
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
                element = line[76:78].strip() if len(line) > 76 else atom_name[0]
                records.append({
                    'record': record_type, 'serial': atom_serial,
                    'atom_name': atom_name, 'residue': residue_name,
                    'chain': chain_id, 'resnum': residue_num,
                    'x': x, 'y': y, 'z': z, 'element': element
                })
            except (ValueError, IndexError):
                continue
    return pd.DataFrame(records)

# Parse the PDB file (use embedded minimal data if download failed)
if pdb_text:
    atoms = parse_pdb(pdb_text)
else:
    # Minimal embedded dataset for offline use
    data = {'record':['ATOM']*5,'serial':[1,2,3,4,5],
            'atom_name':['N','CA','C','O','CB'],
            'residue':['LYS','LYS','LYS','LYS','LYS'],
            'chain':['A']*5,'resnum':[1]*5,
            'x':[3.0,4.4,5.7,5.8,4.6],'y':[6.0,6.0,5.2,4.0,7.4],'z':[0.0,0.0,0.0,0.0,-1.2],
            'element':['N','C','C','O','C']}
    atoms = pd.DataFrame(data)
    print("Using minimal embedded data — download was unavailable.")

print(f"Parsed {len(atoms)} atoms from the PDB file")
print(f"\nFirst 5 rows:")
print(atoms.head())
print(f"\nUnique residues: {atoms['residue'].nunique()}")
print(f"Unique chains: {list(atoms['chain'].unique())}")
print(f"Total atoms: {len(atoms)}")


---
## Part 2 — Exploring Protein Composition

### 2.1 Atom type distribution

Let's look at the elemental composition of the protein. Proteins are primarily made of four elements: carbon (C), nitrogen (N), oxygen (O), and sulfur (S). Hydrogen atoms are often omitted from crystal structures (too small to resolve by X-ray), but are added back computationally before MD simulation.


In [ ]:
# Count atoms by element (ATOM records only = protein atoms; HETATM = small molecules)
protein_atoms = atoms[atoms['record'] == 'ATOM'].copy()
hetatm_atoms  = atoms[atoms['record'] == 'HETATM'].copy()

element_counts = protein_atoms['element'].value_counts()
print("Protein atom counts by element:")
print(element_counts.to_string())
print(f"\nTotal protein atoms: {len(protein_atoms)}")
if len(hetatm_atoms) > 0:
    print(f"HETATM entries (water, ligands, ions): {len(hetatm_atoms)}")
    print("Unique HETATM residues:", hetatm_atoms['residue'].unique())


In [ ]:
# Visualize element distribution as a pie chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart of element composition
colors = {'C': '#4C8CBF', 'N': '#5BA85E', 'O': '#C94040', 'S': '#E8B84B',
          'CA': '#9B59B6', 'MG': '#27AE60'}
pie_colors = [colors.get(e, '#AAAAAA') for e in element_counts.index]

axes[0].pie(element_counts.values, labels=element_counts.index,
            colors=pie_colors, autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 12})
axes[0].set_title(f'Atom Composition of {pdb_id}\n(protein atoms only)', fontweight='bold')

# Bar chart of residue frequency (which amino acids appear most often?)
if len(protein_atoms) > 5:
    residue_counts = protein_atoms[protein_atoms['atom_name'] == 'CA']['residue'].value_counts()
    top_residues = residue_counts.head(10)
    axes[1].bar(top_residues.index, top_residues.values, color='#2E5EAA', alpha=0.85)
    axes[1].set_xlabel('Amino Acid (3-letter code)', fontsize=11)
    axes[1].set_ylabel('Count in Structure', fontsize=11)
    axes[1].set_title(f'10 Most Frequent Residues in {pdb_id}', fontweight='bold')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('protein_composition.png', dpi=150, bbox_inches='tight')
plt.show()


### ✏️ ANNOTATION REQUIRED

1. Which element is most abundant in the protein? Does this make sense given what you know about the chemical composition of amino acids?
2. Look at the most frequent residues. Do you recognize any from the Protease Primer (Supplement S1)? Name one and explain its chemical properties.
3. The pie chart does not include hydrogen atoms. If hydrogens were included, which element would become most abundant? (Hint: every N–H and O–H in the protein has at least one H; the backbone alone has one N–H per residue.)


### My Annotation for Section 2:

*[YOUR ANSWER HERE]*


---
## Part 3 — Calculating Atomic Distances

### 3.1 The distance formula in 3D

To identify hydrogen bonds computationally, you need to calculate distances between atoms. In 3D space:

**d = √[(x₂−x₁)² + (y₂−y₁)² + (z₂−z₁)²]**

This is simply the 3D version of the Pythagorean theorem. The units are Angstroms (Å).

For hydrogen bonds:
- Donor–Acceptor distance must be ≤ 3.5 Å (typical cutoff)
- Donor–H–Acceptor angle must be ≥ 120°

In this section, we will calculate all distances between selected atoms to identify potential hydrogen bond partners.


In [ ]:
def distance_3d(row1, row2):
    """Calculate the 3D Euclidean distance between two atoms (in Angstroms)."""
    dx = row2['x'] - row1['x']
    dy = row2['y'] - row1['y']
    dz = row2['z'] - row1['z']
    return np.sqrt(dx**2 + dy**2 + dz**2)

# Example: find all backbone N and O atoms in the first 10 residues
if len(protein_atoms) > 100:
    first10 = protein_atoms[protein_atoms['resnum'] <= 10]
    backbone_N = first10[first10['atom_name'] == 'N']
    backbone_O = first10[first10['atom_name'] == 'O']

    print(f"Backbone N atoms in residues 1-10: {len(backbone_N)}")
    print(f"Backbone O atoms in residues 1-10: {len(backbone_O)}")

    # Calculate all N-to-O distances (potential H-bond contacts)
    contacts = []
    for _, n_atom in backbone_N.iterrows():
        for _, o_atom in backbone_O.iterrows():
            dist = distance_3d(n_atom, o_atom)
            if dist <= 3.5:  # H-bond distance cutoff
                contacts.append({
                    'donor_res':   f"{n_atom['residue']}{n_atom['resnum']}",
                    'acceptor_res':f"{o_atom['residue']}{o_atom['resnum']}",
                    'N_atom':  n_atom['atom_name'],
                    'O_atom':  o_atom['atom_name'],
                    'distance_A':  round(dist, 3)
                })

    contact_df = pd.DataFrame(contacts).sort_values('distance_A')
    print(f"\nN–O contacts ≤ 3.5 Å in residues 1–10:")
    print(contact_df.to_string(index=False))
else:
    print("Using minimal dataset — distance analysis requires full PDB structure.")
    print("This cell will run fully when the PDB download is available.")


### 3.2 Distance distribution: all Cα–Cα pairs

The α-carbon (Cα) is the central backbone carbon in every amino acid. Looking at Cα–Cα distances tells us about the compactness of the protein and which residues are spatially close (even if far apart in sequence). Residues within 8 Å of each other are generally considered to be in spatial contact.

This is the same type of analysis you will do in Phase 2 — but instead of Cα, you will measure distances between inhibitor atoms and protease active site atoms.


In [ ]:
if len(protein_atoms) > 100:
    ca_atoms = protein_atoms[protein_atoms['atom_name'] == 'CA'].copy()
    ca_coords = ca_atoms[['x','y','z']].values

    # Calculate all pairwise Cα-Cα distances efficiently using broadcasting
    diff = ca_coords[:, np.newaxis, :] - ca_coords[np.newaxis, :, :]
    dist_matrix = np.sqrt((diff**2).sum(axis=-1))

    # Get upper triangle (avoid duplicates)
    upper = dist_matrix[np.triu_indices(len(ca_atoms), k=1)]

    # Plot distribution
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(upper, bins=60, color='#2E5EAA', alpha=0.8, edgecolor='white')
    axes[0].axvline(8.0, color='#CC3300', linestyle='--', linewidth=2, label='8 Å contact cutoff')
    axes[0].set_xlabel('Cα–Cα Distance (Å)', fontsize=11)
    axes[0].set_ylabel('Count (pairs)', fontsize=11)
    axes[0].set_title('Distribution of All Cα–Cα Distances', fontweight='bold')
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3)

    # Contact map (2D heatmap of distances)
    im = axes[1].imshow(dist_matrix, cmap='viridis_r', aspect='auto',
                        vmin=0, vmax=40)
    axes[1].set_xlabel('Residue Number', fontsize=11)
    axes[1].set_ylabel('Residue Number', fontsize=11)
    axes[1].set_title(f'Cα Distance Map ({pdb_id})', fontweight='bold')
    plt.colorbar(im, ax=axes[1], label='Distance (Å)')

    plt.tight_layout()
    plt.savefig('distance_map.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"Number of Cα atoms: {len(ca_atoms)}")
    print(f"Number of Cα pairs within 8 Å: {np.sum(upper <= 8.0)}")
    print(f"Minimum Cα-Cα distance: {upper.min():.2f} Å  (adjacent residues ~3.8 Å)")
    print(f"Maximum Cα-Cα distance: {upper.max():.2f} Å  (protein diameter)")
else:
    print("Full structure needed for distance map visualization.")


### ✏️ ANNOTATION REQUIRED

1. What is the most common Cα–Cα distance in the histogram? What does this tell you about how most residues in the protein are arranged relative to each other?
2. Look at the distance map (heatmap). The bright diagonal regions off-center represent residues that are close in space despite being far in sequence. What structural features of proteins (secondary structures) would create these patterns?
3. **Connection to Phase 2:** In your HTLV-1 protease simulations, you will measure the distance between inhibitor atoms and specific protease residues. Based on the H-bond distance cutoff (3.5 Å) and the contact cutoff (8 Å), which cutoff is more appropriate for identifying direct molecular interactions vs. broader proximity? Explain your reasoning.


### My Annotation for Section 3:

*[YOUR ANSWER HERE]*


---
## Part 4 — B-factors: A Crystal Structure's Hint About Flexibility

### 4.1 What are B-factors?

In a PDB file, every atom has a **B-factor** (also called the temperature factor or Debye-Waller factor). It represents how much that atom's position varies across many unit cells in the crystal — larger B-factor means more positional uncertainty, which correlates with higher **flexibility**.

This is directly relevant to your research: the B-factor from the crystal structure gives a first-pass estimate of which protein regions are flexible vs. rigid. Flexible regions (high B-factor) are likely to show larger RMSD values in your MD simulations.


In [ ]:
if len(protein_atoms) > 100 and 'bfactor' not in protein_atoms.columns:
    # Parse B-factors from the original PDB text
    bfactors = []
    for line in pdb_text.split('\n'):
        if line.startswith('ATOM'):
            try:
                bfactor = float(line[60:66])
                atom_name = line[12:16].strip()
                resnum = int(line[22:26])
                bfactors.append({'resnum': resnum, 'atom_name': atom_name, 'bfactor': bfactor})
            except (ValueError, IndexError):
                continue
    bf_df = pd.DataFrame(bfactors)

    # Average B-factor per residue (use CA atom as representative)
    ca_bfactors = bf_df[bf_df['atom_name'] == 'CA'].groupby('resnum')['bfactor'].mean().reset_index()

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.fill_between(ca_bfactors['resnum'], ca_bfactors['bfactor'],
                    alpha=0.4, color='#2E5EAA')
    ax.plot(ca_bfactors['resnum'], ca_bfactors['bfactor'],
            color='#2E5EAA', linewidth=1.5)

    # Mark high flexibility regions (B-factor > 1.5× mean)
    mean_bf = ca_bfactors['bfactor'].mean()
    high_flex = ca_bfactors[ca_bfactors['bfactor'] > 1.5 * mean_bf]
    ax.scatter(high_flex['resnum'], high_flex['bfactor'],
               color='#CC3300', s=40, zorder=5, label=f'High flexibility (>1.5× mean)')
    ax.axhline(mean_bf, color='gray', linestyle='--', alpha=0.7, label=f'Mean B-factor ({mean_bf:.1f} Å²)')

    ax.set_xlabel('Residue Number', fontsize=12)
    ax.set_ylabel('B-factor (Å²)', fontsize=12)
    ax.set_title(f'Per-Residue B-factors: {pdb_id} Lysozyme', fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('bfactors.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"Mean B-factor: {mean_bf:.2f} Å²")
    print(f"Residues with high flexibility (>{1.5*mean_bf:.1f} Å²):")
    print(high_flex['resnum'].values)
else:
    print("B-factor analysis will run with full PDB download.")


### ✏️ ANNOTATION REQUIRED

1. Which regions of lysozyme (by residue number) show the highest B-factors? These are likely loop regions — unstructured segments between helices and sheets.
2. **Prediction for Phase 2:** The HTLV-1 protease polyproline loop (residues 91–100) is known to be flexible. Based on what you learned about B-factors today, what B-factor values would you expect for these residues compared to the core of the protein?
3. B-factors come from static crystal structures. MD simulations provide dynamic information. What specific information would an MD simulation give you about loop flexibility that a B-factor *cannot* tell you? (Think about what changes over time in a simulation.)


### My Annotation for Section 4:

*[YOUR ANSWER HERE]*


---
## ✅ Notebook 2 Completion Checklist

- [ ] All code cells have been run
- [ ] All **ANNOTATION REQUIRED** sections completed
- [ ] All plots display correctly (composition, distance distribution, distance map, B-factors)
- [ ] Notebook saved and downloaded as .ipynb for submission

---
## 🔗 Connection to Future Notebooks

In **Notebook 3**, you will load an actual MD simulation trajectory of lysozyme and calculate RMSD over time — the key metric for measuring how much the protein moves during a simulation. You will discover which regions are most flexible, and see whether the B-factors from today's crystal structure predicted the right regions.

In **Notebooks 4 and 5**, you will apply these same tools to the HTLV-1 protease — your actual research system.
